<a href="https://colab.research.google.com/github/dipendrahub/Dipendra_Sharma_Kafle/blob/main/RAG_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retrieval-Augmented Generation (RAG): Practical

In this practical you build a small RAG system that answers questions about a **fictional university** (Lumora University). The university is invented, so a language model cannot know the answers from its training. Any correct answer has to come from the documents (external source).

A RAG system has three parts:

| Part | What it does | Your task |
|---|---|---|
| **1. Retrieval** | Finds the documents most relevant to the question | Exercise 1 |
| **2. Augmentation** | Adds the retrieved documents to the prompt | Exercise 2 |
| **3. Generation** | The language model writes the answer from the prompt | Exercise 3 |
| **Extra** | Refuse to answer when nothing relevant is retrieved | Exercise 4 |
| **Extra** | Limits of keyword retrieval, and chunking | Exercise 5 |

At the end, you compare the answers **with RAG** and **without RAG**.



## Setup

In [3]:
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.feature_extraction.text import TfidfVectorizer

# Load a small language model (Flan-T5 small). It is downloaded automatically (about 300 MB).
MODEL_NAME = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

def generate(prompt, max_new_tokens=60):
    """Send a prompt to the language model and return its answer as text."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(output[0], skip_special_tokens=True)

print("Model ready.")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model ready.


## The knowledge base
Ten short documents about Lumora University. Each document is treated as one unit that can be retrieved.

In [4]:
documents = {
"Library": """The Lumora University Library is located in the Harbor Building. It is open from 8:00 to 22:00 on weekdays and from 10:00 to 18:00 on weekends. Students can borrow up to 12 books at a time for a loan period of 21 days. Each book can be renewed twice online through the student portal. Late returns are charged a fine of 0.30 euros per day for each book. The library also offers 40 silent study desks, which can be booked up to 3 days in advance. Group study rooms are available on the second floor and hold up to 6 people.""",

"Admissions and Fees": """Applications for the autumn intake at Lumora University must be submitted by 15 March through the online application portal. Applicants need a bachelor degree certificate, a CV and two reference letters. The tuition fee is 2,400 euros per semester and must be paid within 30 days after enrollment. The Aurora Grant gives 1,000 euros to students who finished their previous degree with a grade point average above 3.7. Applications for the grant are handled by the Admissions Office in Room 210. Late applications are accepted until 1 April for a surcharge of 100 euros.""",

"Housing": """Lumora University offers rooms in Kestrel Hall, a dormitory located five minutes from the main campus. The rent is 380 euros per month, and a deposit of 500 euros is due at signing. Quiet hours are from 22:00 to 07:00 every day. Residents may host guests for a maximum of 2 consecutive nights and must register them at the front desk. Each floor has a shared kitchen and a laundry room. Rooms are assigned by lottery in July, and students must apply for housing before 30 June. Smoking is not allowed anywhere in the building.""",

"Exams": """At Lumora University, grades are given on a scale from 0 to 20, and a course is passed with a grade of 10 or higher. Students may attempt each exam a maximum of two times. A second attempt requires a re-exam fee of 50 euros. Exam results are published within 14 days on the student portal. Students who disagree with a result can submit an appeal to the Examination Board within 7 days of publication. During exams, only a basic calculator and a university ID are allowed. Phones must be switched off and left in bags at the front of the room.""",

"IT Services": """The IT Helpdesk at Lumora University is in Room 105 of the Harbor Building and can be reached by phone at extension 4455. Students connect to the campus Wi-Fi network named Lumora-Secure with their university login. A forgotten password can be reset at any time through the Helpdesk portal using the recovery email address. Every student receives 200 GB of cloud storage and 100 free printed pages per semester. Extra pages cost 0.05 euros each. Software such as MATLAB and Office is available for free to enrolled students through the portal.""",

"Student Clubs": """More than 30 student clubs are active at Lumora University. The Robotics Club meets every Thursday at 18:00 in Lab B12, and the Chess Club meets every Tuesday at 17:30 in the Student Lounge. New members are welcome at any meeting and membership is free. To found a new club, students need at least 10 members and a faculty sponsor, and must register the club with the Student Council before the end of October. Registered clubs receive a yearly budget of 300 euros and can book rooms for free. The club fair takes place in the first week of each semester.""",

"Cafeteria": """The campus cafeteria, called The Heron, is open from 7:30 to 15:00 on weekdays and is closed on weekends. A standard lunch menu costs 4.50 euros and includes a main dish, a side and a drink. A vegetarian menu is offered every day, and a vegan option is available on Mondays and Wednesdays. Payment is only possible with the Lumora meal card, which can be topped up at the kiosk near the entrance. Coffee and snacks are sold at the small counter until 17:00. Allergy information is displayed next to every dish.""",

"Career Center": """The Career Center is located on the first floor of the Atlas Building. Master students must complete an internship of at least 8 weeks before they can submit their thesis. The center organizes a CV workshop every second Wednesday and mock interviews by appointment. The annual career fair takes place in November, with over 60 companies attending. Students can browse internship offers on the career portal and book a one-to-one advice session of 30 minutes. The center also keeps a list of companies that have hosted interns from Lumora University in the past.""",

"Health and Wellbeing": """Lumora University offers free counseling for students who feel overwhelmed by pressure, anxiety or personal difficulties. Each student can have up to 6 sessions per semester at no cost, booked through the wellbeing portal. The counseling office is in Room 118 of the Atlas Building. A nurse is available in Room 105 of the Atlas Building every weekday from 9:00 to 16:00. In an emergency, call 112, and for campus security call extension 4000, which is staffed day and night. A quiet room for rest and meditation is open in the library basement.""",

"Campus Transport": """The campus has free bike parking in front of the Harbor Building and the Atlas Building, with covered racks available near the dormitory. Bikes can be repaired at a small workshop every Friday afternoon. The shuttle bus line S2 runs between Central Station and the main campus every 20 minutes from 6:30 to 21:00. Students ride the shuttle for free by showing their university card. Parking for cars requires a permit that costs 120 euros per year, and permits are limited to 400 spaces. Electric scooters may be used on campus roads but must be parked at the marked areas.""",
}

titles = list(documents.keys())
texts = list(documents.values())
print(len(documents), "documents loaded")

10 documents loaded


## Part 1: Retrieval
The retrieval step finds the documents that best match the question.

Each document is converted into a **TF-IDF vector**: a list of numbers that describes which words the document contains, giving more weight to words that are rare across the collection. The question is converted into a vector in the same way. The documents whose vectors are most similar to the question vector (cosine similarity) are retrieved.

In [5]:
# Build the index: one TF-IDF vector per document
vectorizer = TfidfVectorizer(stop_words="english")
doc_vectors = vectorizer.fit_transform(texts)

print("Index shape (documents x vocabulary words):", doc_vectors.shape)

Index shape (documents x vocabulary words): (10, 330)


### EXERCISE 1: Write the retrieval function
TF-IDF vectors have length 1, so the cosine similarity between a question and a document is the **dot product** of their vectors. The function returns the `k` documents with the highest scores.

In [6]:
def retrieve(question, k=2):
    """Return the k most relevant documents as a list of (title, text, score)."""
    q_vector = vectorizer.transform([question])          # 1. convert the question into a vector
    scores = doc_vectors @ q_vector.T                    # 2. dot product of all document vectors with q_vector
    scores = np.asarray(scores.todense()).ravel()        #    (flatten to a 1-D array, one score per document)
    top_ids = np.argsort(-scores, kind="stable")[:k]     # 3. indices of the k highest scores, best first
    return [(titles[i], texts[i], scores[i]) for i in top_ids]

In [8]:
# Try it on a few questions
for question in ["How much is the rent for the Kestrel Hall dormitory?",
                 "When does the Robotics Club meet?"]:
    print("Question:", question)
    for title, text, score in retrieve(question, k=2):
        print(f"   {score:.2f}  {title}")
    print()

Question: How much is the rent for the Kestrel Hall dormitory?
   0.31  Housing
   0.05  Campus Transport

Question: When does the Robotics Club meet?
   0.48  Student Clubs
   0.00  Library



## Part 2: Augmentation
The augmentation step builds the **prompt** that is sent to the language model. The prompt contains three things:
1. an instruction,
2. the retrieved documents as context,
3. the question.

### EXERCISE 2: Write the prompt-building function

In [9]:
def build_prompt(question, retrieved):
    """Build the prompt from the question and the retrieved documents."""
    # 1. join the texts of the retrieved documents into one string, separated by blank lines
    context = "\n\n".join(text for title, text, score in retrieved)

    prompt = (
        "Answer the question using only the context below. "
        "If the answer is not in the context, say \"I don't know\".\n\n"
        f"Context:\n{context}\n\n"
        # 2. add the question, followed by "Answer:"
        f"Question: {question}\nAnswer:"
    )
    return prompt

In [11]:
# Look at a complete prompt
q = "When does the Robotics Club meet?"
print(build_prompt(q, retrieve(q, k=2)))

Answer the question using only the context below. If the answer is not in the context, say "I don't know".

Context:
More than 30 student clubs are active at Lumora University. The Robotics Club meets every Thursday at 18:00 in Lab B12, and the Chess Club meets every Tuesday at 17:30 in the Student Lounge. New members are welcome at any meeting and membership is free. To found a new club, students need at least 10 members and a faculty sponsor, and must register the club with the Student Council before the end of October. Registered clubs receive a yearly budget of 300 euros and can book rooms for free. The club fair takes place in the first week of each semester.

The Lumora University Library is located in the Harbor Building. It is open from 8:00 to 22:00 on weekdays and from 10:00 to 18:00 on weekends. Students can borrow up to 12 books at a time for a loan period of 21 days. Each book can be renewed twice online through the student portal. Late returns are charged a fine of 0.30 e

## Part 3: Generation
The generation step sends the prompt to the language model, which writes the answer. The `generate` function from the setup does this.

### EXERCISE 3: Put the three parts together

In [12]:
def rag_answer(question, k=2):
    """Answer a question with RAG: retrieve, augment, generate."""
    retrieved = retrieve(question, k)          # 1. retrieve the k most relevant documents
    prompt = build_prompt(question, retrieved) # 2. build the prompt
    answer = generate(prompt)                  # 3. generate the answer from the prompt
    sources = [title for title, text, score in retrieved]
    return answer, sources

In [28]:
answer, sources = rag_answer("When does the Robotics Club meet?")
print("Answer :", answer)
print("Sources:", sources)

Answer : Thursday at 18:00
Sources: ['Student Clubs', 'Library']


## Comparison: with RAG and without RAG
The same model answers the same questions in two ways:
- **Without RAG:** the question is sent directly to the model.
- **With RAG:** the question goes through retrieval, augmentation and generation.

The last question is **not covered** by any document.

In [14]:
questions = [
    "How much is the rent for the Kestrel Hall dormitory?",
    "Until what time is the library open on weekdays?",
    "When does the Robotics Club meet?",
    "How many free pages can I print each semester?",
    "What time does the swimming pool open at Lumora University?",   # not in the documents
]

rows = []
for question in questions:
    answer_rag, sources = rag_answer(question)
    rows.append({
        "Question": question,
        "Without RAG": generate(question),
        "With RAG": answer_rag,
        "Sources used": ", ".join(sources),
    })

pd.set_option("display.max_colwidth", None)
pd.DataFrame(rows)

,Question,Without RAG,With RAG,Sources used
0,How much is the rent for the Kestrel Hall dormitory?,$45,380 euros,"Housing, Campus Transport"
1,Until what time is the library open on weekdays?,6:00 p.m.,8:00 to 22:00,"Library, Health and Wellbeing"
2,When does the Robotics Club meet?,"September 27, 2018",Thursday at 18:00,"Student Clubs, Library"
3,How many free pages can I print each semester?,240,100,"IT Services, Student Clubs"
4,What time does the swimming pool open at Lumora University?,6:00 p.m.,8:00,"Library, IT Services"


### Questions
1. For which questions does RAG give a correct answer? What does the model answer without RAG?
2. Look at the question about the swimming pool. What does the model answer with RAG? What did retrieval return, and why?
3. Write two questions of your own and test them in the cell below.

In [31]:
my_question = "...."   # TODO: write your own question

answer, sources = rag_answer(my_question)
print("Without RAG:", generate(my_question))
print("With RAG   :", answer)
print("Sources    :", sources)

Without RAG: ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ...
With RAG   : I don't know
Sources    : ['Library', 'Admissions and Fees']


## Exercise 4: Refuse to answer when retrieval finds nothing relevant
In the comparison, the swimming pool question was not covered by any document, but retrieval still returned two documents. The **retrieval score** shows how well the best document matches the question. A low score means that nothing relevant was found.

First, look at the best score for each question.

In [15]:
for question in questions:
    best_title, best_text, best_score = retrieve(question, k=1)[0]
    print(f"{best_score:.2f}  {best_title:15s}  {question}")

0.31  Housing          How much is the rent for the Kestrel Hall dormitory?
0.31  Library          Until what time is the library open on weekdays?
0.48  Student Clubs    When does the Robotics Club meet?
0.35  IT Services      How many free pages can I print each semester?
0.19  Library          What time does the swimming pool open at Lumora University?


### EXERCISE 4: Write a safer RAG function
Write `rag_answer_safe`. If the best retrieval score is below `threshold`, it returns `"I don't know"` and does not call the language model. Otherwise it answers as `rag_answer` does.

In [16]:
def rag_answer_safe(question, k=2, threshold=0.25):
    """RAG that refuses to answer when the best retrieved document scores below the threshold."""
    retrieved = retrieve(question, k)
    best_score = retrieved[0][2]          # 1. the score of the best retrieved document
    if best_score < threshold:            # 2. the best score is below the threshold
        return "I don't know", []
    prompt = build_prompt(question, retrieved)
    answer = generate(prompt)
    sources = [title for title, text, score in retrieved]
    return answer, sources

In [17]:
# CHECK 4
answer, sources = rag_answer_safe("What time does the swimming pool open at Lumora University?")
assert answer == "I don't know" and sources == [], "the swimming pool question should be refused"
answer, sources = rag_answer_safe("When does the Robotics Club meet?")
assert len(sources) > 0, "questions covered by the documents should still be answered"
print("Check 4 passed.")

Check 4 passed.


In [18]:
for question in questions:
    answer, sources = rag_answer_safe(question)
    print("Question:", question)
    print("Answer  :", answer)
    print("Sources :", sources, "\n")

Question: How much is the rent for the Kestrel Hall dormitory?
Answer  : 380 euros
Sources : ['Housing', 'Campus Transport'] 

Question: Until what time is the library open on weekdays?
Answer  : 8:00 to 22:00
Sources : ['Library', 'Health and Wellbeing'] 

Question: When does the Robotics Club meet?
Answer  : Thursday at 18:00
Sources : ['Student Clubs', 'Library'] 

Question: How many free pages can I print each semester?
Answer  : 100
Sources : ['IT Services', 'Student Clubs'] 

Question: What time does the swimming pool open at Lumora University?
Answer  : I don't know
Sources : [] 



### Questions
1. Which questions are answered and which are refused with `threshold=0.25`?
2. Set `threshold=0.35` and run the cell again. What changes? Why is choosing the threshold a trade-off?

## Exercise 5: Limits of keyword retrieval, and chunking
Our retrieval has two limitations. Part A covers the first, Part B the second.

### Part A: Keyword retrieval only matches words
TF-IDF compares the **words** in the question with the words in the documents. It does not understand meaning. Look at what happens when the question uses a different word than the document.

In [19]:
question = "Where can I leave my bicycle?"

for title, text, score in retrieve(question, k=2):
    print(f"{score:.2f}  {title}")
print()
answer, sources = rag_answer(question)
print("Answer:", answer)

0.00  Library
0.00  Admissions and Fees

Answer: I don't know


The Campus Transport document contains the answer, but it says **"bike"**, not "bicycle". The two words do not match, so the score is 0.

### EXERCISE 5A: Expand the question with synonyms
Write `expand_question`. It adds the synonym of every word in the question that appears in the `synonyms` dictionary.

In [20]:
synonyms = {"bicycle": "bike", "test": "exam"}

def expand_question(question, synonyms):
    """Append the synonyms of the question's words to the question."""
    words = question.lower().replace("?", "").split()
    extra = [synonyms[word] for word in words if word in synonyms]   # synonyms of the words in the question
    return question + " " + " ".join(extra)

In [21]:
# CHECK 5A
expanded = expand_question("Where can I leave my bicycle?", synonyms)
assert "bike" in expanded, "the synonym 'bike' should be added to the question"
assert expand_question("Where is the library?", synonyms).strip() == "Where is the library?"
print("Check 5A passed.")

Check 5A passed.


In [22]:
for question in ["Where can I leave my bicycle?", "How many times can I take a test?"]:
    print("Question:", question)
    print("  before:", [(title, round(float(score), 2)) for title, text, score in retrieve(question, k=2)])
    expanded = expand_question(question, synonyms)
    print("  after :", [(title, round(float(score), 2)) for title, text, score in retrieve(expanded, k=2)])
    print()

Question: Where can I leave my bicycle?
  before: [('Library', 0.0), ('Admissions and Fees', 0.0)]
  after : [('Campus Transport', 0.14), ('Library', 0.0)]

Question: How many times can I take a test?
  before: [('Exams', 0.14), ('Library', 0.0)]
  after : [('Exams', 0.41), ('Library', 0.0)]



**Questions**
1. What changed after the question was expanded?
2. Why does a hand-written synonym list not work for a real system? What kind of method could match questions and documents by **meaning** instead of exact words?

### Part B: Chunking
So far, each retrieved document is about 100 words long, and the prompt contains **two whole documents** even though the answer is a single sentence. The language model can only read a limited amount of text (512 tokens for this model), and extra text can distract it.

**Chunking** splits each document into smaller pieces, called chunks. Each chunk is indexed and retrieved separately. Consecutive chunks **overlap**, so that a sentence cut at the border still appears completely in one chunk.

### EXERCISE 5B: Write the chunking function
Split a text into chunks of `chunk_size` words. Each chunk starts `chunk_size - overlap` words after the previous one.

In [23]:
def chunk_text(text, chunk_size=30, overlap=10):
    """Split a text into chunks of chunk_size words, with `overlap` words shared between neighbours."""
    words = text.split()
    step = chunk_size - overlap              # 1. how many words to move forward for each new chunk
    chunks = []
    for start in range(0, len(words), step):
        chunk_words = words[start:start + chunk_size]   # 2. the chunk_size words that begin at `start`
        chunks.append(" ".join(chunk_words))
        if start + chunk_size >= len(words): # the end of the text is reached
            break
    return chunks

In [24]:
# CHECK 5B
test = chunk_text(" ".join(str(i) for i in range(100)), chunk_size=40, overlap=10)
assert len(test) == 3, f"expected 3 chunks, got {len(test)}"
assert test[0].split()[-10:] == test[1].split()[:10], "neighbouring chunks must share 10 words"
print("Check 5B passed.")

Check 5B passed.


Now we split all documents into chunks and build a new index. Each chunk remembers the document it comes from.

In [25]:
chunk_titles, chunk_texts = [], []
for title, text in documents.items():
    for chunk in chunk_text(text, chunk_size=30, overlap=10):
        chunk_titles.append(title)
        chunk_texts.append(chunk)

chunk_vectorizer = TfidfVectorizer(stop_words="english")
chunk_vectors = chunk_vectorizer.fit_transform(chunk_texts)
print(len(documents), "documents ->", len(chunk_texts), "chunks")

def retrieve_chunks(question, k=2):
    """Same as retrieve, but searches chunks instead of whole documents."""
    q_vector = chunk_vectorizer.transform([question])
    scores = np.asarray((chunk_vectors @ q_vector.T).todense()).ravel()
    top_ids = np.argsort(-scores, kind="stable")[:k]
    return [(chunk_titles[i], chunk_texts[i], scores[i]) for i in top_ids]

10 documents -> 50 chunks


In [26]:
question = "When does the Robotics Club meet?"

docs_prompt = build_prompt(question, retrieve(question, k=2))
chunks_prompt = build_prompt(question, retrieve_chunks(question, k=2))

print("Retrieved chunks:")
for title, text, score in retrieve_chunks(question, k=2):
    print(f"  {score:.2f}  [{title}] {text}")

print()
print("Prompt length with whole documents:", len(docs_prompt.split()), "words")
print("Prompt length with chunks         :", len(chunks_prompt.split()), "words")
print()
print("Answer with whole documents:", generate(docs_prompt))
print("Answer with chunks         :", generate(chunks_prompt))

Retrieved chunks:
  0.41  [Student Clubs] More than 30 student clubs are active at Lumora University. The Robotics Club meets every Thursday at 18:00 in Lab B12, and the Chess Club meets every Tuesday at 17:30
  0.26  [Student Clubs] meeting and membership is free. To found a new club, students need at least 10 members and a faculty sponsor, and must register the club with the Student Council before

Prompt length with whole documents: 231 words
Prompt length with chunks         : 89 words

Answer with whole documents: Thursday at 18:00
Answer with chunks         : 18:00


**Questions**
1. How much shorter is the prompt with chunks? Does the retrieved text still contain the answer?
2. Change `chunk_size` in the chunking cell to 10, then to 100, and run the cells again. What happens to the retrieved text and to the answer? Why is the chunk size a trade-off?

## Summary
- **Retrieval** selects the documents most similar to the question.
- **Augmentation** places those documents (external source) in the prompt, together with an instruction and the question.
- **Generation** lets the language model answer from the prompt.
- Without retrieval, the model has no access to the documents and cannot answer correctly.
- Retrieval always returns the top-k documents, even when none of them is relevant. The quality of the answer depends on the quality of retrieval.
- A score threshold lets the system refuse to answer when nothing relevant is found. The threshold is a trade-off between refusing valid questions and answering without support.
- Keyword retrieval (TF-IDF) matches words, not meaning, so a question and a document with different words can fail to match. Embeddings address this.
- **Chunking** splits documents into smaller pieces, so that the prompt contains only the relevant text. The chunk size is a trade-off between precision and context.